# Topic: Command Injection Prevention - OS Command Execution risks, subprocess configuration, and argument array arrays
 
## 1. THE OS COMMAND INJECTION VULNERABILITY
- **OS Command Injection**: Occurs when an application passes unsanitized user inputs directly into an OS system shell execution (e.g., executing system commands like `ping`, `ls`, or `curl`).
  - If an attacker passes inputs containing shell metacharacters (like `;`, `&`, `|`, or backticks `` ` ``), they can chain arbitrary system commands (e.g., `ping 127.0.0.1; cat /etc/passwd`), compromise the server, and execute arbitrary code.
 
## 2. SUBPROCESS RULES: shell=True VS shell=False
- Python's standard **`subprocess`** module is used to run external commands.
- **DANGEROUS (`shell=True`)**:
  - Spawns a shell intermediary (e.g., `/bin/sh` or `cmd.exe`) to execute the string command.
  - User input strings are interpreted as raw shell commands, making injection trivial.
- **SECURE (`shell=False` - Default)**:
  - Executes the binary target directly.
  - Requires arguments to be passed as a **list of strings** (e.g., `["ping", "-c", "1", host]`).
  - The OS kernel processes the arguments strictly as parameter inputs for the target binary, ignoring any shell metacharacters.
 
---


In [ ]:
import subprocess
import shlex

# 1. Vulnerable execution using shell=True
def vulnerable_ping(host):
    """Executes a ping command using shell=True (Vulnerable!)."""
    # DANGEROUS: String interpolation with shell=True
    cmd = f"ping -c 1 {host}"
    print(f"[EXECUTING SHELL CMD]: {cmd}")
    
    try:
        output = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT)
        return output.decode()
    except subprocess.CalledProcessError as e:
        return f"Error: {e.output.decode()}"

print("--- Vulnerable Command Execution (Normal Input) ---")
# Ping localhost once
print(vulnerable_ping("127.0.0.1"))



In [ ]:
print("\n--- Command Injection Attack ---")
# Attacker inputs: 127.0.0.1; echo "HACKED"
# The shell runs: ping -c 1 127.0.0.1; echo "HACKED"
malicious_host = "127.0.0.1; echo 'SYSTEM COMPROMISED'"
print(vulnerable_ping(malicious_host))
# Expected: Runs ping, and then executes the injected echo command!



In [ ]:
# 2. Secure execution using shell=False
def secure_ping(host):
    """Executes a ping command securely using shell=False (Default)."""
    # Pass command and args as a list of strings
    cmd_list = ["ping", "-c", "1", host]
    print(f"[SECURE EXECUTING BINARY]: {cmd_list}")
    
    try:
        # shell=False is default
        output = subprocess.check_output(cmd_list, shell=False, stderr=subprocess.STDOUT)
        return output.decode()
    except subprocess.CalledProcessError as e:
        return f"Error: {e.output.decode()}"

print("\n--- Secure Command Execution ---")
# Try to run malicious injection on secure ping
print(secure_ping(malicious_host))
# Expected: Returns an error from the ping binary stating that the host is unknown,
# preventing the shell from executing "echo 'SYSTEM COMPROMISED'"!
